# 实验三 · RGB → BGR 通道重排（结构化访存）

**所属**：《并行计算》第三章 · ARM NEON SIMD 编程　|　**难度**：⭐⭐ 进阶　|　**预计时长**：20–30 分钟

> **实验说明**
> 1. 本实验采用分步实现的方式：由 **v1 串行版本**起，依次引入 **v2 NEON 结构化访问**与 **v3 循环展开**，每一版本均实际编译并运行，据此观察性能的逐步变化。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖 **ARM(aarch64/arm64) + NEON**；请在华为鲲鹏处理器上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. 三个版本的代码为递增关系：后一版本在前一版本基础上新增一个实现方式，输出表格相应增加一行，便于对照阅读，理解优化的引入过程。
> 6. 本实验为图像处理系列的第一个案例，重点掌握 **`vld3`/`vst3` 结构化访存**，为后续需要“拆分通道后再计算”的算法（实验四、五、六）奠定基础。

## 🎯 学习目标

完成本实验后，学生应能够：

- 理解彩色图像的**交织存储**（AoS：R0 G0 B0 R1 G1 B1 …）
- 掌握**结构化访存指令** `vld3q_u8` / `vst3q_u8`，理解其一步完成**解交织 / 交织**（AoS↔SoA）的机制
- 认识一类“**纯访存、零计算**”的负载，理解其加速比通常有限的原因
- 理解**原地（in-place）操作**在性能测量中的处理方法（借助奇数次循环保证最终状态正确）
- 为后续“解交织 → 对通道计算 → 交织写回”的图像算法（实验四、五、六）建立基础

## 🗺️ 学习路径

1. **准备阶段**：理解图像的交织存储，以及“通道重排”这一纯访存任务
2. **v1 · 串行基准**：实现 `rgb_to_bgr_serial_no_vec`（关闭向量化，作为基准）与 `rgb_to_bgr_serial`（允许自动向量化）
   → 考察编译器对逐字节交换操作的自动向量化能力
3. **v2 · NEON 结构化访存**：新增 `rgb_to_bgr_neon`，使用 `vld3`/`vst3` 解交织与交织
   → 掌握结构化访存的核心用法
4. **v3 · 循环展开 + 预取**：新增 `rgb_to_bgr_neon_unroll`
   → 考察在纯访存负载上循环展开与预取的效果
5. **可视化与分析**：以 v3 的输出结果绘制加速比柱状图，分析纯访存负载的性能特征

## 1. 背景与动机

彩色图像在内存中通常采用**交织存储**：`R0 G0 B0 R1 G1 B1 …`，即同一像素的三个通道相邻存放（此即 **AoS**，Array of Structures）。

RGB→BGR 转换只是**交换 R 与 B 两个通道**，不涉及任何算术运算，是一个纯粹的**内存搬运与重排**任务。它本身较为简单，但由此引出的 **`vld3`/`vst3`** 指令是整个图像处理的基础：只有先将交织数据**拆分为分离的通道**（**SoA**，Structure of Arrays），才能对各个通道分别施加不同的运算——这正是后续白平衡、灰度、YUV 等算法所依赖的前提。

## 2. 算法与原理

对每个像素执行：`(R, G, B) → (B, G, R)`，即交换第 0 通道与第 2 通道。

- **标量实现**：逐像素、逐字节地交换，需借助临时变量。
- **NEON 实现**：`vld3q_u8` 一次读取 16 个像素（48 字节），并由硬件**自动解交织**为 R、G、B 三个向量寄存器；此时只需**交换 R 与 B 两个寄存器**（寄存器间交换，开销极低），再用 `vst3q_u8` 交织写回原内存位置。

> 本案例为**原地（in-place）操作**：读入与写回使用同一块内存。由此带来一个测量上的细节——每执行一次转换，数据在 RGB 与 BGR 之间翻转一次。为保证多次计时循环后数据仍处于 BGR 状态（以便与参考结果比对），程序将循环次数 `NTIMES` 设为**奇数（21）**，使翻转的总次数为奇数。

## 3. 核心 NEON 指令与技巧

```c
for (; i <= num_pixels - 16; i += 16) {
  uint8x16x3_t rgb = vld3q_u8(data + i * 3);  // 解交织：16 像素 → R/G/B 三向量
  uint8x16_t temp = rgb.val[0];               // 交换 R(val[0]) 与 B(val[2])
  rgb.val[0] = rgb.val[2];
  rgb.val[2] = temp;
  vst3q_u8(data + i * 3, rgb);                // 交织写回：SoA → AoS
}
```

- **`vld3q_u8`**：结构化加载，硬件自动将 `RGBRGB…` 拆分为 R、G、B 三个向量（AoS → SoA）
- **`uint8x16x3_t`**：包含 3 个 `uint8x16_t` 的结构，分别对应三个通道
- **`vst3q_u8`**：结构化存储，将三个通道向量交织写回内存（SoA → AoS）
- **本案例不含任何计算**：加速完全来自 `vld3`/`vst3` 的高效搬运与更少的指令数
- **`__builtin_prefetch`**（v3）：提前预取后续缓存行，用以缓解访存延迟

## 4. 环境准备

In [ ]:
import platform, subprocess, shutil, sys

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
IS_ARM = platform.machine().lower() in ("aarch64", "arm64", "armv7l", "armv8l")
if not IS_ARM:
    print("\n⚠️  当前不是 ARM 架构，NEON 代码无法在此编译运行。")
elif CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
else:
    print("\n✅ 环境就绪：ARM 架构 + 编译器可用，可以开始实验！")

In [ ]:
import subprocess, platform, re, shutil

MACHINE = platform.machine().lower()


def compile_c(src, out):
    """尝试多组编译参数，返回可执行文件名；失败则打印错误。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    if MACHINE in ("armv7l", "armv8l"):  # 32 位 ARM 需显式开 NEON
        flagsets = ["-O3 -fPIC -mfpu=neon -mfloat-abi=hard -march=armv7-a"]
    else:  # aarch64 / arm64：NEON 默认开启
        flagsets = ["-O3 -fPIC"]
    last = ""
    for fl in flagsets:
        cmd = f"{base} {fl} {src} -o {out} -lm"
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode == 0:
            print("✅ 编译成功：", cmd)
            return out
        last = r.stderr
    print("❌ 编译失败：\n", last)
    return None


def run_bin(out, *args):
    """运行可执行文件并打印其输出。"""
    r = subprocess.run(
        [f"./{out}"] + [str(a) for a in args], capture_output=True, text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
    return r.stdout


def parse_table(text):
    """解析 | 方法 | 耗时 | 加速比 | 校验 | 表格，兼容 '4.21 x' 与 '4.21x'。"""
    rows = []
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 3:
            continue
        name = cells[0]
        if name.lower() in ("method", "方法") or set(name) <= set("-: "):
            continue
        mt = re.search(r"[-+]?\d*\.?\d+", cells[1])
        ms = re.search(r"[-+]?\d*\.?\d+", cells[2])
        if not mt:
            continue
        rows.append(
            {
                "method": name,
                "time": float(mt.group()),
                "speedup": float(ms.group()) if ms else None,
            }
        )
    return rows


In [ ]:
import matplotlib.pyplot as plt


def plot_speedup(rows, title=""):
    rows = [r for r in rows if r["speedup"] is not None]
    if not rows:
        print("未解析到可绘制的加速比。")
        return
    names = [r["method"] for r in rows]
    sp = [r["speedup"] for r in rows]
    best = sp.index(max(sp))
    colors = ["#9aa0a6" if s <= 1.05 else "#295E96" for s in sp]
    colors[best] = "#C7000B"  # 最快版本标红
    plt.figure(figsize=(8, 4))
    bars = plt.bar(names, sp, color=colors)
    plt.axhline(1.0, ls="--", c="gray", lw=1)
    for b, s in zip(bars, sp):
        plt.text(
            b.get_x() + b.get_width() / 2,
            s,
            f"{s:.2f}x",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    plt.ylabel("Speedup (x)")
    plt.title(title)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
# 创建源代码目录
!mkdir -p src_rgb2bgr

## 5. v1 · 串行基准实现

与前两个实验一致，第一个版本包含**两个函数体相同**的串行实现，区别仅在于是否允许编译器自动向量化：

<table>
  <thead>
    <tr>
      <th style="text-align: left;">函数</th>
      <th style="text-align: left;">说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><code>rgb_to_bgr_serial_no_vec</code></td>
      <td style="text-align: left;">通过 <code>no-tree-vectorize</code> <strong>显式关闭</strong>自动向量化，作为<strong>性能基准</strong>（1.00×）</td>
    </tr>
    <tr>
      <td style="text-align: left;"><code>rgb_to_bgr_serial</code></td>
      <td style="text-align: left;">源码相同，但<strong>允许编译器自动向量化</strong></td>
    </tr>
  </tbody>
</table>

### 💡 关注点
RGB→BGR 是逐字节的数据交换，请在运行后观察 `Serial (Auto)` 相对基准的加速比，考察编译器能否对这种带跨步（stride-3）的交换操作自动向量化。

### 数据布局与原地操作
- `data`：交织存储的图像缓冲，长度为 `width × height × 3` 字节；转换**原地**进行
- 参考结果 `dst_ref` 由基准内核对原图翻转**一次**得到（即标准的 BGR）
- 计时循环执行 `NTIMES = 21` 次（**奇数**），确保循环结束后数据处于 BGR 状态，从而可用 `check_diff` 与参考结果比对；每次计时前用 `memcpy` 将缓冲复位为原图，复位不计入计时

In [ ]:
%%writefile src_rgb2bgr/rgb2bgr_v1.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

// Use an odd number of loops so the final in-place result is flipped (BGR)
#define NTIMES 21

double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

const char* check_diff(const uint8_t* ref, const uint8_t* test, long n) {
  for (long i = 0; i < n * 3; i++) {
    if (ref[i] != test[i]) {
      return "FAIL";
    }
  }
  return "PASS";
}

// ----------------------------------------------------------------------------
// 1. Serial Version (Forced No-Vectorization) - Baseline (In-place)
// Processes pixel by pixel (AoS layout); swap R (index 0) and B (index 2)
// ----------------------------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void rgb_to_bgr_serial_no_vec(uint8_t* restrict data, long num_pixels) {
  for (long i = 0; i < num_pixels; i++) {
    uint8_t temp = data[i * 3 + 0];
    data[i * 3 + 0] = data[i * 3 + 2];
    data[i * 3 + 2] = temp;
  }
}

// ----------------------------------------------------------------------------
// 2. Serial Version (Compiler may auto-vectorize) (In-place)
// ----------------------------------------------------------------------------
void rgb_to_bgr_serial(uint8_t* restrict data, long num_pixels) {
  for (long i = 0; i < num_pixels; i++) {
    // Swap R (index 0) and B (index 2)
    uint8_t temp = data[i * 3 + 0];
    data[i * 3 + 0] = data[i * 3 + 2];
    data[i * 3 + 2] = temp;
  }
}

int main(int argc, char** argv) {
  if (argc != 3) {
    printf("Usage: %s <width> <height>\n", argv[0]);
    return 1;
  }
  int width = atoi(argv[1]);
  int height = atoi(argv[2]);

  printf("===========================================================\n");
  printf(" RGB2BGR v1: Serial Baseline and Auto-Vectorized Serial (In-place)\n");
  printf(" Image: %d x %d\n", width, height);
  printf(" Loops: %d (Odd number to ensure final state is BGR)\n", NTIMES);
  printf("===========================================================\n");

  long num_pixels = (long)width * height;
  size_t bytes = num_pixels * 3 * sizeof(uint8_t);
  bytes = (bytes + 15) & ~(size_t)15;

  // Aligned allocation (16-byte) for SIMD friendliness
  uint8_t* src = (uint8_t*)aligned_alloc(16, bytes);
  uint8_t* dst_ref = (uint8_t*)aligned_alloc(16, bytes);
  uint8_t* data_test = (uint8_t*)aligned_alloc(16, bytes);

  if (!src || !dst_ref || !data_test) {
    printf("Error: Allocation failed\n");
    return 1;
  }

  // Initialize with dummy RGB values
  for (long i = 0; i < bytes; i++) {
    src[i] = (uint8_t)(i % 256);
  }

  // Golden reference = ONE flip (RGB -> BGR) by the baseline kernel
  memcpy(dst_ref, src, bytes);
  rgb_to_bgr_serial_no_vec(dst_ref, num_pixels);

  double start, end;

  // Report header
  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");

  // Baseline: Serial (No-Vec). NTIMES is odd, so after the timing loop the
  // buffer is flipped an odd number of times => final state is BGR.
  memcpy(data_test, src, bytes);  // reset before timing
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) rgb_to_bgr_serial_no_vec(data_test, num_pixels);
  end = get_time_ms();
  double t_no_vec = (end - start) / NTIMES;
  const char* s_no_vec = check_diff(dst_ref, data_test, num_pixels);
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |  %-4s |\n", t_no_vec, s_no_vec);

  // Serial (Auto-Vec)
  memcpy(data_test, src, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) rgb_to_bgr_serial(data_test, num_pixels);
  end = get_time_ms();
  double t_serial = (end - start) / NTIMES;
  const char* s_serial = check_diff(dst_ref, data_test, num_pixels);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_serial,
         t_no_vec / t_serial, s_serial);
  printf("-------------------------------------------------\n");

  free(src);
  free(dst_ref);
  free(data_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_rgb2bgr/rgb2bgr_v1.c", "src_rgb2bgr/rgb2bgr_v1")
out_v1 = run_bin(BIN, 1920, 1080)

## 6. v2 · NEON 结构化访存实现

在 v1 的基础上**新增 `rgb_to_bgr_neon` 函数**，使用 `vld3`/`vst3` 完成解交织与交织：

```c
uint8x16x3_t rgb = vld3q_u8(data + i * 3);  // 解交织：16 像素 → R/G/B
uint8x16_t temp = rgb.val[0];               // 交换 R 与 B 两个向量
rgb.val[0] = rgb.val[2];
rgb.val[2] = temp;
vst3q_u8(data + i * 3, rgb);                // 交织写回
```

### 🔑 知识点
- **结构化访存是 `vld3`/`vst3` 的核心价值**：单条指令即可完成 AoS 与 SoA 之间的转换，而无需手工进行复杂的移位与掩码操作
- **通道交换在寄存器层面完成**：解交织后，R 与 B 已分处不同寄存器，交换二者仅是寄存器重命名，开销极低
- **纯访存负载**：本内核不含任何算术运算，其性能完全取决于访存效率

运行后，请观察 NEON 相对基准的加速比，并与后续“有计算”的图像案例（实验四、五）作对比。

In [ ]:
%%writefile src_rgb2bgr/rgb2bgr_v2.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

// Use an odd number of loops so the final in-place result is flipped (BGR)
#define NTIMES 21

double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

const char* check_diff(const uint8_t* ref, const uint8_t* test, long n) {
  for (long i = 0; i < n * 3; i++) {
    if (ref[i] != test[i]) {
      return "FAIL";
    }
  }
  return "PASS";
}

// ----------------------------------------------------------------------------
// 1. Serial Version (Forced No-Vectorization) - Baseline (In-place)
// Processes pixel by pixel (AoS layout); swap R (index 0) and B (index 2)
// ----------------------------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void rgb_to_bgr_serial_no_vec(uint8_t* restrict data, long num_pixels) {
  for (long i = 0; i < num_pixels; i++) {
    uint8_t temp = data[i * 3 + 0];
    data[i * 3 + 0] = data[i * 3 + 2];
    data[i * 3 + 2] = temp;
  }
}

// ----------------------------------------------------------------------------
// 2. Serial Version (Compiler may auto-vectorize) (In-place)
// ----------------------------------------------------------------------------
void rgb_to_bgr_serial(uint8_t* restrict data, long num_pixels) {
  for (long i = 0; i < num_pixels; i++) {
    // Swap R (index 0) and B (index 2)
    uint8_t temp = data[i * 3 + 0];
    data[i * 3 + 0] = data[i * 3 + 2];
    data[i * 3 + 2] = temp;
  }
}

// ----------------------------------------------------------------------------
// 3. NEON Optimization: Structured Access vld3/vst3 (In-place)
//   vld3q_u8: load 16 RGB pixels and de-interleave into R,G,B (AoS->SoA)
//   swap register 0 (R) and 2 (B); vst3q_u8: interleave back (SoA->AoS)
// ----------------------------------------------------------------------------
void rgb_to_bgr_neon(uint8_t* restrict data, long num_pixels) {
  long i = 0;
  // Process 16 pixels (48 bytes) per iteration
  for (; i <= num_pixels - 16; i += 16) {
    // 1. Load and de-interleave (AoS -> SoA)
    uint8x16x3_t rgb = vld3q_u8(data + i * 3);

    // 2. Swap R and B channels in registers (SoA modification)
    uint8x16_t temp = rgb.val[0];
    rgb.val[0] = rgb.val[2];
    rgb.val[2] = temp;

    // 3. Interleave and store back to the exact same memory (SoA -> AoS)
    vst3q_u8(data + i * 3, rgb);
  }

  // Scalar cleanup for remaining pixels
  for (; i < num_pixels; i++) {
    uint8_t temp = data[i * 3 + 0];
    data[i * 3 + 0] = data[i * 3 + 2];
    data[i * 3 + 2] = temp;
  }
}

int main(int argc, char** argv) {
  if (argc != 3) {
    printf("Usage: %s <width> <height>\n", argv[0]);
    return 1;
  }
  int width = atoi(argv[1]);
  int height = atoi(argv[2]);

  printf("===========================================================\n");
  printf(" RGB2BGR v2: Add NEON Structured Access vld3/vst3 (In-place)\n");
  printf(" Image: %d x %d\n", width, height);
  printf(" Loops: %d (Odd number to ensure final state is BGR)\n", NTIMES);
  printf("===========================================================\n");

  long num_pixels = (long)width * height;
  size_t bytes = num_pixels * 3 * sizeof(uint8_t);
  bytes = (bytes + 15) & ~(size_t)15;

  // Aligned allocation (16-byte) for SIMD friendliness
  uint8_t* src = (uint8_t*)aligned_alloc(16, bytes);
  uint8_t* dst_ref = (uint8_t*)aligned_alloc(16, bytes);
  uint8_t* data_test = (uint8_t*)aligned_alloc(16, bytes);

  if (!src || !dst_ref || !data_test) {
    printf("Error: Allocation failed\n");
    return 1;
  }

  // Initialize with dummy RGB values
  for (long i = 0; i < bytes; i++) {
    src[i] = (uint8_t)(i % 256);
  }

  // Golden reference = ONE flip (RGB -> BGR) by the baseline kernel
  memcpy(dst_ref, src, bytes);
  rgb_to_bgr_serial_no_vec(dst_ref, num_pixels);

  double start, end;

  // Report header
  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");

  // Baseline: Serial (No-Vec). NTIMES is odd, so after the timing loop the
  // buffer is flipped an odd number of times => final state is BGR.
  memcpy(data_test, src, bytes);  // reset before timing
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) rgb_to_bgr_serial_no_vec(data_test, num_pixels);
  end = get_time_ms();
  double t_no_vec = (end - start) / NTIMES;
  const char* s_no_vec = check_diff(dst_ref, data_test, num_pixels);
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |  %-4s |\n", t_no_vec, s_no_vec);

  // Serial (Auto-Vec)
  memcpy(data_test, src, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) rgb_to_bgr_serial(data_test, num_pixels);
  end = get_time_ms();
  double t_serial = (end - start) / NTIMES;
  const char* s_serial = check_diff(dst_ref, data_test, num_pixels);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_serial,
         t_no_vec / t_serial, s_serial);

  // NEON (vld3/vst3 structured access)
  memcpy(data_test, src, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) rgb_to_bgr_neon(data_test, num_pixels);
  end = get_time_ms();
  double t_neon = (end - start) / NTIMES;
  const char* s_neon = check_diff(dst_ref, data_test, num_pixels);
  printf("| NEON Intrinsic  | %9.3f | %5.2f x |  %-4s |\n", t_neon,
         t_no_vec / t_neon, s_neon);
  printf("-------------------------------------------------\n");

  free(src);
  free(dst_ref);
  free(data_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_rgb2bgr/rgb2bgr_v2.c", "src_rgb2bgr/rgb2bgr_v2")
out_v2 = run_bin(BIN, 1920, 1080)

## 7. v3 · 循环展开与数据预取

在 v2 的基础上**新增 `rgb_to_bgr_neon_unroll` 函数**：单次迭代处理 **32 个像素**（两个 16 像素块），并加入数据预取。

```c
for (; i <= num_pixels - 32; i += 32) {
  __builtin_prefetch(data + i * 3 + 192);   // 预取后续缓存行
  // 块 0：vld3 → 交换 R/B → vst3
  // 块 1：vld3 → 交换 R/B → vst3
}
```

### 🔑 知识点
- **循环展开**：单次迭代处理更多像素，减少循环控制开销，并使多组访存操作可以重叠
- **数据预取 `__builtin_prefetch`**：提示处理器提前将后续数据载入缓存，以隐藏内存访问延迟；对访存受限的负载可能有一定帮助

> 需要注意的是，对于 RGB→BGR 这类**纯访存**负载，其性能瓶颈在于内存带宽而非计算或指令调度，因此循环展开与预取所能带来的收益往往有限。

至此，**四个实现**（含基准）全部就位，输出表格与完整版源码 `05_neon_rgb_to_bgr.c` 保持一致。

In [ ]:
%%writefile src_rgb2bgr/rgb2bgr_v3.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

// Use an odd number of loops so the final in-place result is flipped (BGR)
#define NTIMES 21

double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

const char* check_diff(const uint8_t* ref, const uint8_t* test, long n) {
  for (long i = 0; i < n * 3; i++) {
    if (ref[i] != test[i]) {
      return "FAIL";
    }
  }
  return "PASS";
}

// ----------------------------------------------------------------------------
// 1. Serial Version (Forced No-Vectorization) - Baseline (In-place)
// Processes pixel by pixel (AoS layout); swap R (index 0) and B (index 2)
// ----------------------------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void rgb_to_bgr_serial_no_vec(uint8_t* restrict data, long num_pixels) {
  for (long i = 0; i < num_pixels; i++) {
    uint8_t temp = data[i * 3 + 0];
    data[i * 3 + 0] = data[i * 3 + 2];
    data[i * 3 + 2] = temp;
  }
}

// ----------------------------------------------------------------------------
// 2. Serial Version (Compiler may auto-vectorize) (In-place)
// ----------------------------------------------------------------------------
void rgb_to_bgr_serial(uint8_t* restrict data, long num_pixels) {
  for (long i = 0; i < num_pixels; i++) {
    // Swap R (index 0) and B (index 2)
    uint8_t temp = data[i * 3 + 0];
    data[i * 3 + 0] = data[i * 3 + 2];
    data[i * 3 + 2] = temp;
  }
}

// ----------------------------------------------------------------------------
// 3. NEON Optimization: Structured Access vld3/vst3 (In-place)
//   vld3q_u8: load 16 RGB pixels and de-interleave into R,G,B (AoS->SoA)
//   swap register 0 (R) and 2 (B); vst3q_u8: interleave back (SoA->AoS)
// ----------------------------------------------------------------------------
void rgb_to_bgr_neon(uint8_t* restrict data, long num_pixels) {
  long i = 0;
  // Process 16 pixels (48 bytes) per iteration
  for (; i <= num_pixels - 16; i += 16) {
    // 1. Load and de-interleave (AoS -> SoA)
    uint8x16x3_t rgb = vld3q_u8(data + i * 3);

    // 2. Swap R and B channels in registers (SoA modification)
    uint8x16_t temp = rgb.val[0];
    rgb.val[0] = rgb.val[2];
    rgb.val[2] = temp;

    // 3. Interleave and store back to the exact same memory (SoA -> AoS)
    vst3q_u8(data + i * 3, rgb);
  }

  // Scalar cleanup for remaining pixels
  for (; i < num_pixels; i++) {
    uint8_t temp = data[i * 3 + 0];
    data[i * 3 + 0] = data[i * 3 + 2];
    data[i * 3 + 2] = temp;
  }
}

// ----------------------------------------------------------------------------
// 4. NEON Unrolled: process 32 pixels per loop + prefetch (In-place)
// ----------------------------------------------------------------------------
void rgb_to_bgr_neon_unroll(uint8_t* restrict data, long num_pixels) {
  long i = 0;
  // Process 32 pixels (96 bytes) per iteration
  for (; i <= num_pixels - 32; i += 32) {
    // Prefetch next cache lines
    __builtin_prefetch(data + i * 3 + 192);

    // Block 0
    uint8x16x3_t rgb0 = vld3q_u8(data + i * 3);
    uint8x16_t temp0 = rgb0.val[0];
    rgb0.val[0] = rgb0.val[2];
    rgb0.val[2] = temp0;
    vst3q_u8(data + i * 3, rgb0);

    // Block 1
    uint8x16x3_t rgb1 = vld3q_u8(data + i * 3 + 48);
    uint8x16_t temp1 = rgb1.val[0];
    rgb1.val[0] = rgb1.val[2];
    rgb1.val[2] = temp1;
    vst3q_u8(data + i * 3 + 48, rgb1);
  }

  // Scalar cleanup
  for (; i < num_pixels; i++) {
    uint8_t temp = data[i * 3 + 0];
    data[i * 3 + 0] = data[i * 3 + 2];
    data[i * 3 + 2] = temp;
  }
}

int main(int argc, char** argv) {
  if (argc != 3) {
    printf("Usage: %s <width> <height>\n", argv[0]);
    return 1;
  }
  int width = atoi(argv[1]);
  int height = atoi(argv[2]);

  printf("===========================================================\n");
  printf(" RGB2BGR v3: Add NEON Unrolled with Prefetch (In-place)\n");
  printf(" Image: %d x %d\n", width, height);
  printf(" Loops: %d (Odd number to ensure final state is BGR)\n", NTIMES);
  printf("===========================================================\n");

  long num_pixels = (long)width * height;
  size_t bytes = num_pixels * 3 * sizeof(uint8_t);
  bytes = (bytes + 15) & ~(size_t)15;

  // Aligned allocation (16-byte) for SIMD friendliness
  uint8_t* src = (uint8_t*)aligned_alloc(16, bytes);
  uint8_t* dst_ref = (uint8_t*)aligned_alloc(16, bytes);
  uint8_t* data_test = (uint8_t*)aligned_alloc(16, bytes);

  if (!src || !dst_ref || !data_test) {
    printf("Error: Allocation failed\n");
    return 1;
  }

  // Initialize with dummy RGB values
  for (long i = 0; i < bytes; i++) {
    src[i] = (uint8_t)(i % 256);
  }

  // Golden reference = ONE flip (RGB -> BGR) by the baseline kernel
  memcpy(dst_ref, src, bytes);
  rgb_to_bgr_serial_no_vec(dst_ref, num_pixels);

  double start, end;

  // Report header
  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");

  // Baseline: Serial (No-Vec). NTIMES is odd, so after the timing loop the
  // buffer is flipped an odd number of times => final state is BGR.
  memcpy(data_test, src, bytes);  // reset before timing
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) rgb_to_bgr_serial_no_vec(data_test, num_pixels);
  end = get_time_ms();
  double t_no_vec = (end - start) / NTIMES;
  const char* s_no_vec = check_diff(dst_ref, data_test, num_pixels);
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |  %-4s |\n", t_no_vec, s_no_vec);

  // Serial (Auto-Vec)
  memcpy(data_test, src, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) rgb_to_bgr_serial(data_test, num_pixels);
  end = get_time_ms();
  double t_serial = (end - start) / NTIMES;
  const char* s_serial = check_diff(dst_ref, data_test, num_pixels);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_serial,
         t_no_vec / t_serial, s_serial);

  // NEON (vld3/vst3 structured access)
  memcpy(data_test, src, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) rgb_to_bgr_neon(data_test, num_pixels);
  end = get_time_ms();
  double t_neon = (end - start) / NTIMES;
  const char* s_neon = check_diff(dst_ref, data_test, num_pixels);
  printf("| NEON Intrinsic  | %9.3f | %5.2f x |  %-4s |\n", t_neon,
         t_no_vec / t_neon, s_neon);

  // NEON Unrolled (32 pixels per loop + prefetch)
  memcpy(data_test, src, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) rgb_to_bgr_neon_unroll(data_test, num_pixels);
  end = get_time_ms();
  double t_unroll = (end - start) / NTIMES;
  const char* s_unroll = check_diff(dst_ref, data_test, num_pixels);
  printf("| NEON Unrolled   | %9.3f | %5.2f x |  %-4s |\n", t_unroll,
         t_no_vec / t_unroll, s_unroll);
  printf("-------------------------------------------------\n");

  free(src);
  free(dst_ref);
  free(data_test);
  return 0;
}

In [ ]:
BIN = compile_c("src_rgb2bgr/rgb2bgr_v3.c", "src_rgb2bgr/rgb2bgr_v3")
out_v3 = run_bin(BIN, 1920, 1080)

## 8. 📈 性能可视化（基于 v3 的四版本结果）

v3 的输出包含全部四个实现的耗时与加速比，据此绘制柱状图，以完整呈现各手段的效果。
（灰色表示无明显加速，蓝色表示存在加速，**红色标示性能最优的版本**）

In [ ]:
rows_v3 = parse_table(out_v3)
for r in rows_v3:
    print(f'{r["method"]:16s} {r["time"]:9.3f} ms   {r["speedup"]:.2f}x')
plot_speedup(rows_v3, "RGB to BGR: performance of four implementations (1920x1080)")

## 9. 结果分析

> 注：具体数值随硬件平台、图像尺寸、编译器版本与系统负载而变化，请以本机实际运行结果为准；下述分析针对数据所反映的**趋势与规律**。

RGB→BGR 的结果通常呈现以下特征：

**① 手写 NEON 相对基准有明显加速。**

实测中 `NEON Intrinsic` 可达 2× 以上。原因在于：RGB→BGR 是**带跨步（stride-3）的字节重排**，编译器难以自动生成结构化访存指令；而 `vld3`/`vst3` 由硬件一步完成解交织与交织，指令数大幅减少，因而收益显著。

需要说明的是，本例属于**纯访存型负载**（不含任何算术运算），其性能最终受限于内存带宽，因此加速比不会随向量宽度线性增长。

**② `vld3`/`vst3` 的意义不在于本例的加速幅度，而在于它提供的能力。**

结构化访存指令一步完成 AoS 与 SoA 的转换。在本例中它仅用于交换通道，作用尚不明显；但在后续需要**对各通道分别计算**的算法中，“先解交织、再计算、后交织”成为标准范式，`vld3`/`vst3` 的价值才真正体现。

**③ 循环展开与预取的收益有限。**

这类手段主要用于隐藏计算延迟或提高指令级并行；而纯访存负载的瓶颈在带宽，因此其效果通常不显著。这再次说明：**优化手段需与瓶颈相匹配。**

---

### 🎓 结论
RGB→BGR 让我们认识了**结构化访存 `vld3`/`vst3`** 与**纯访存型负载**的性能特征。它是图像处理系列的入门案例：本身简单，却提供了后续所有图像算法都要用到的关键工具。

> 一个引导性的问题：如果在解交织之后**顺便对通道施加运算**（例如各通道乘以不同的增益），`vld3`/`vst3` 的价值便会真正显现——这正是**实验四 白平衡**所要完成的工作。

## 10. 🔧 动手练习

请修改代码、重新编译并运行，观察性能的变化（建议先独立完成，再阅读思考题）：

1. 将 `vld3q_u8` / `vst3q_u8` 改为 `vld4q_u8` / `vst4q_u8`，实现 RGBA→BGRA（4 通道）的转换。
2. 以较大图像（如 `4096 4096`）运行，估算各实现所达到的有效内存带宽（GB/s = 处理字节数 / 耗时）。
3. 删除 v3 中的 `__builtin_prefetch` 语句后重新运行，观察预取对本负载的实际影响。
4. 【进阶】为 `compile_c` 的编译参数添加 `-march=native` 后重新运行，观察 `Serial (Auto)` 是否有所改善。

## 11. 🤔 思考题

- `vld3`/`vst3` 的真正用途是什么？仅仅是为了交换通道顺序吗？
- 为何计时循环的次数 `NTIMES` 要取奇数？若取偶数会导致什么问题？
- 对纯访存型负载，循环展开与数据预取为何收益有限？
- 原地操作与非原地操作在性能测量上各有何注意事项？

## 12. 小结与后续

本实验完成了 RGB→BGR 从串行到 NEON 的实现过程：

<table>
  <thead>
    <tr>
      <th style="text-align: left;">版本</th>
      <th style="text-align: left;">新增内容</th>
      <th style="text-align: left;">涉及知识点</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><strong>v1</strong></td>
      <td style="text-align: left;"><code>rgb_to_bgr_serial_no_vec</code> + <code>rgb_to_bgr_serial</code></td>
      <td style="text-align: left;">性能基准、交织存储、原地操作</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v2</strong></td>
      <td style="text-align: left;"><code>rgb_to_bgr_neon</code></td>
      <td style="text-align: left;">结构化访存 <code>vld3</code>/<code>vst3</code>、AoS↔SoA</td>
    </tr>
    <tr>
      <td style="text-align: left;"><strong>v3</strong></td>
      <td style="text-align: left;"><code>rgb_to_bgr_neon_unroll</code></td>
      <td style="text-align: left;">循环展开、数据预取</td>
    </tr>
  </tbody>
</table>

通过 RGB→BGR，我们掌握了**结构化访存 `vld3`/`vst3`**，并认识了**纯访存型负载**的性能特征——其加速比受内存带宽约束，各种计算侧优化收效有限。

➡️ **后续内容：实验四 白平衡（Gray World）**。我们将在解交织之后真正“对通道施加运算”，把 `vld3`/`vst3` 与**规约统计、定点运算、饱和处理**结合起来，构成第一个综合性的图像算法。